# Supply-chain research training on Colab Pro (A100 / V100)

Runbook for the cost-and-carbon NSGA-II + LSTM components on Colab Pro (A100 40 GB or V100 16 GB depending on quota, 24 h wall-clock cap).

Reference docs:
* `cloud_training/README_CLOUD_SETUP.md` for the platform comparison.
* `cloud_training/TRAINING_GUIDE.md` for the ordered runbook across NSGA-II x 10 seeds, LSTM, PPO 1M-step.

Checkpoints are written under `/content/drive/MyDrive/supply_chain_checkpoints/` so the same notebook can be resumed after a disconnect via `--resume <drive-path>`.

In [ ]:
# Cell (b): GPU sanity check
!nvidia-smi

In [ ]:
# Cell (c): clone the repo and install pinned requirements
import os
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/example/Supply-chain.git')
REPO_REF = os.environ.get('REPO_REF', 'main')
if not os.path.isdir('/content/Supply-chain'):
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} /content/Supply-chain
%cd /content/Supply-chain
!pip install --quiet -r supply_chain_research/requirements.txt

In [ ]:
# Cell (d): mount Google Drive and copy the preprocessed
# .npy / .parquet artefacts into the local working tree.
from google.colab import drive
drive.mount('/content/drive')
import shutil
from pathlib import Path
DRIVE_DATA = Path('/content/drive/MyDrive/supply_chain_data')
TARGET_DIR = Path('/content/Supply-chain/data/processed')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_DATA.exists():
    for src in DRIVE_DATA.glob('*.npy'):
        shutil.copy2(src, TARGET_DIR / src.name)
    for src in DRIVE_DATA.glob('*.parquet'):
        shutil.copy2(src, TARGET_DIR / src.name)
    print('Copied:', sorted(p.name for p in TARGET_DIR.iterdir()))
else:
    print('No Drive folder; the runner will fall back to the '
          'synthetic generators in supply_chain_research.')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/supply_chain_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell (e): NSGA-II x 10 seeds (resumable via Drive)
!python cloud_training/local_training_runner.py --component nsga2 --seeds 10 --resume /content/drive/MyDrive/supply_chain_checkpoints/nsga2.pkl

In [ ]:
# Cell (f): LSTM demand forecaster
!python cloud_training/local_training_runner.py --component lstm --resume /content/drive/MyDrive/supply_chain_checkpoints/lstm.pt

In [ ]:
# Cell (g): summarise the artefacts produced this session
import json
from pathlib import Path
RESULTS = Path('/content/Supply-chain/data/results')
for path in sorted(RESULTS.glob('*')):
    print(f'{path.name:40s} {path.stat().st_size:>12} bytes')
metrics = RESULTS / 'lstm_metrics.json'
if metrics.exists():
    print('\nLSTM metrics:')
    print(json.dumps(json.loads(metrics.read_text()), indent=2))